AI MCP Version 2

In [ ]:
%run ~/data/env.py
! cat ~/data/env.py

In [ ]:
from openai import OpenAI
from mcp.client.sse import sse_client
from mcp import ClientSession

import json
import os

client = OpenAI(
    api_key=OPENAI_API_KEY,
    base_url=AI_BASE_URL,
)

MCP_URL = "http://localhost:8000/sse"

MCP-Tools automatisch in OpenAI-Tools übersetzen:

In [ ]:
async def get_openai_tools_from_mcp():
    async with sse_client(MCP_URL) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            mcp_tools = await session.list_tools()

            openai_tools = []

            for tool in mcp_tools.tools:
                openai_tools.append({
                    "type": "function",
                    "name": tool.name,
                    "description": tool.description or f"MCP tool {tool.name}",
                    "parameters": tool.inputSchema or {
                        "type": "object",
                        "properties": {},
                        "additionalProperties": False,
                    },
                })

            return openai_tools

MCP-Tool ausführen:

In [ ]:
async def call_mcp_tool(tool_name: str, arguments: dict | None = None):
    async with sse_client(MCP_URL) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            return await session.call_tool(tool_name, arguments or {})

MCP-Resultat in Text wandeln:

In [ ]:
def mcp_result_to_text(result) -> str:
    if hasattr(result, "content"):
        parts = []

        for item in result.content:
            if hasattr(item, "text"):
                parts.append(item.text)
            else:
                parts.append(str(item))

        return "\n\n".join(parts)

    return str(result)

Kompletter OpenAI-Loop mit automatisch gelesenen MCP-Tools:

In [ ]:
async def ask_with_mcp(prompt: str):
    tools = await get_openai_tools_from_mcp()

    response = client.responses.create(
        model=AI_MODEL,
        input=prompt,
        tools=tools,
    )

    tool_outputs = []

    for item in response.output:
        if item.type != "function_call":
            continue

        arguments = json.loads(item.arguments or "{}")

        mcp_result = await call_mcp_tool(
            tool_name=item.name,
            arguments=arguments,
        )

        tool_outputs.append({
            "type": "function_call_output",
            "call_id": item.call_id,
            "output": mcp_result_to_text(mcp_result),
        })

    if not tool_outputs:
        return response.output_text

    final_response = client.responses.create(
        model=AI_MODEL,
        previous_response_id=response.id,
        input=tool_outputs,
    )

    return final_response.output_text

Verwendung:

In [ ]:
answer = await ask_with_mcp("Liste die Dateien im aktuellen Verzeichnis auf.")
print(answer)